In [ ]:

import pandas as pd
import sys
import os

sys.path.append("..")
from src.preprocess import Preprocessor
from src.data_loader import DataLoader


loader=DataLoader("../data/merged/hotels.csv", "preprocessor")

df = loader.load()


x_train, x_test, y_train, y_test = loader.split(target_column="price_category")

prep = Preprocessor()

x_train_filled, fill_values = prep.fillna_train(df=x_train)
x_test_filled = prep.fillna_test(df=x_test, fill_values=fill_values)


/Users/asadbek/Documents/hotel_price_category/notebooks/../src/preprocess.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(fill_values[col], inplace=True)
/Users/asadbek/Documents/hotel_price_category/notebooks/../src/preprocess.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

# Feature creating

In [2]:
x_train_filled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1006 entries, 1200 to 1126
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1006 non-null   object 
 1   price               1006 non-null   float64
 2   adults              1006 non-null   int64  
 3   city                1006 non-null   object 
 4   distance_to_center  1006 non-null   float64
 5   description         1006 non-null   object 
 6   bf_has_included     1006 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(3)
memory usage: 56.0+ KB


# Distance category

In [3]:
def distance_category(distance):
    if distance < 1:
        return "Very Close"
    elif distance < 3:
        return "Close"
    elif distance < 7:
        return "Medium"
    else:
        return "Far"

x_train_filled["distance_category"] = x_train_filled["distance_to_center"].apply(distance_category)
x_test_filled["distance_category"] = x_test_filled["distance_to_center"].apply(distance_category)

In [4]:
x_train_filled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1006 entries, 1200 to 1126
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1006 non-null   object 
 1   price               1006 non-null   float64
 2   adults              1006 non-null   int64  
 3   city                1006 non-null   object 
 4   distance_to_center  1006 non-null   float64
 5   description         1006 non-null   object 
 6   bf_has_included     1006 non-null   bool   
 7   distance_category   1006 non-null   object 
dtypes: bool(1), float64(2), int64(1), object(4)
memory usage: 63.9+ KB


# City popularity

In [5]:
city_counts = df["city"].value_counts()

x_train_filled["city_popularity"] = x_train_filled["city"].map(city_counts)
x_test_filled["city_popularity"] = x_test_filled["city"].map(city_counts)

In [6]:
x_train_filled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1006 entries, 1200 to 1126
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1006 non-null   object 
 1   price               1006 non-null   float64
 2   adults              1006 non-null   int64  
 3   city                1006 non-null   object 
 4   distance_to_center  1006 non-null   float64
 5   description         1006 non-null   object 
 6   bf_has_included     1006 non-null   bool   
 7   distance_category   1006 non-null   object 
 8   city_popularity     1006 non-null   int64  
dtypes: bool(1), float64(2), int64(2), object(4)
memory usage: 71.7+ KB


# Adults category

In [7]:
def adults_category(n):
    if n == 1:
        return "Solo"
    elif n == 2:
        return "Couple"
    elif n <= 4:
        return "Family"
    else:
        return "Group"

x_train_filled["adults_category"] = x_train_filled["adults"].apply(adults_category)
x_test_filled["adults_category"] = x_test_filled["adults"].apply(adults_category)

In [8]:
x_train_filled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1006 entries, 1200 to 1126
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1006 non-null   object 
 1   price               1006 non-null   float64
 2   adults              1006 non-null   int64  
 3   city                1006 non-null   object 
 4   distance_to_center  1006 non-null   float64
 5   description         1006 non-null   object 
 6   bf_has_included     1006 non-null   bool   
 7   distance_category   1006 non-null   object 
 8   city_popularity     1006 non-null   int64  
 9   adults_category     1006 non-null   object 
dtypes: bool(1), float64(2), int64(2), object(5)
memory usage: 79.6+ KB


In [9]:
from collections import Counter

print("Original class distribution:", Counter(y_train))

Original class distribution: Counter({'Standard': 268, 'Luxury': 257, 'Premium': 255, 'Budget': 226})


In [10]:
x_train_encoded, encoders, one_hot_cols = prep.encode_train(df=x_train_filled)
x_test_encoded = prep.encode_test(
  df=x_test_filled, 
  encoders=encoders, 
  onehot_cols=one_hot_cols, 
  train_columns=x_train_encoded.columns
)



# Skewness

In [11]:
numeric_df = x_train_encoded.select_dtypes(include="number")
skewness = numeric_df.skew()
skewed_features = skewness[(skewness >= 0.4)].sort_values(ascending=False)

In [12]:
skewness

title                 0.035105
price                 1.656087
adults                0.146373
city                 -0.421847
distance_to_center    1.547860
description          -0.615350
distance_category     0.221266
city_popularity      -0.242268
adults_category      -0.308518
dtype: float64

In [13]:
skewed_features

price                 1.656087
distance_to_center    1.547860
dtype: float64

In [14]:
skewed_features.index.tolist()


['price', 'distance_to_center']

In [15]:
import numpy as np

X_train_transformed = x_train_encoded.copy()
X_test_transformed = x_test_encoded.copy()

for col in skewed_features.index.tolist():
    if (X_train_transformed[col] >= 0).all():
        
        X_train_transformed[col] = np.log1p(X_train_transformed[col])
        X_test_transformed[col] = np.log1p(X_test_transformed[col])

In [16]:
numeric_df = X_train_transformed.select_dtypes(include="number")
skewness = numeric_df.skew()
skewed_features = skewness[(skewness >= 0.4)].sort_values(ascending=False)

In [17]:
skewness

title                 0.035105
price                -0.191766
adults                0.146373
city                 -0.421847
distance_to_center    0.223584
description          -0.615350
distance_category     0.221266
city_popularity      -0.242268
adults_category      -0.308518
dtype: float64

In [18]:
skewed_features

Series([], dtype: float64)

In [19]:
x_train_scaled, scalers = prep.scale_train(df=X_train_transformed)
x_test_scaled = prep.scale_test(df=X_test_transformed, scalers=scalers)

# Random OverSampling

In [20]:
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN


ros = RandomOverSampler(random_state=42)
x_ros, y_ros = ros.fit_resample(x_train_scaled, y_train)

/Users/asadbek/Documents/hotel_price_category/myenv/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/asadbek/Documents/hotel_price_category/myenv/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [21]:
x_ros

,title,price,adults,city,distance_to_center,description,bf_has_included,distance_category,city_popularity,adults_category
0,1.452105,-0.539808,-0.371009,-0.848011,1.310374,0.714497,0.352168,0.659246,-1.247003,-0.620669
1,-0.208544,1.738826,0.562079,0.589635,-1.006983,0.280029,0.352168,1.488248,1.136137,0.386417
2,-0.255655,0.161211,0.562079,0.302106,-0.818415,0.714497,0.352168,-0.998758,-1.480984,0.386417
3,1.134108,-0.175779,0.095535,0.014577,-0.084621,0.236582,0.352168,-0.998758,-1.281666,-0.620669
4,0.933888,-0.621425,-0.837553,0.877164,1.367473,0.062795,0.352168,0.659246,0.728837,-1.627756
...,...,...,...,...,...,...,...,...,...,...
1067,-1.480530,0.376989,1.495167,0.589635,0.089421,0.931731,0.352168,-0.998758,1.136137,0.386417
1068,-1.221422,0.219788,-0.371009,0.589635,0.442148,0.714497,0.352168,0.659246,1.136137,-0.620669
1069,1.334328,0.759873,1.028623,1.164693,1.095691,0.671050,0.352168,0.659246,-1.498315,0.386417
1070,1.652325,0.219788,-0.371009,0.589635,1.127899,1.018625,0.352168,0.659246,1.136137,-0.620669


In [22]:
y_ros

0       Standard
1         Luxury
2        Premium
3       Standard
4       Standard
          ...   
1067     Premium
1068     Premium
1069     Premium
1070     Premium
1071     Premium
Name: price_category, Length: 1072, dtype: object

# Training (OVR)

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.metrics import classification_report, accuracy_score

ovr_model = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
ovr_model.fit(x_ros, y_ros)

# Predict and evaluate
y_pred_ovr = ovr_model.predict(x_test_scaled)
print("=== One-vs-Rest (OvR) ===")
print(classification_report(y_test, y_pred_ovr))
acc_ovr_rand=accuracy_score(y_test, y_pred_ovr)


=== One-vs-Rest (OvR) ===
              precision    recall  f1-score   support

      Budget       0.86      0.89      0.87        54
      Luxury       0.88      1.00      0.93        70
     Premium       0.73      0.59      0.65        59
    Standard       0.72      0.71      0.72        69

    accuracy                           0.80       252
   macro avg       0.80      0.80      0.79       252
weighted avg       0.79      0.80      0.80       252



# OVO

In [24]:

ovo_model = OneVsOneClassifier(LogisticRegression(max_iter=1000, random_state=42))
ovo_model.fit(x_ros, y_ros)


y_pred_ovo = ovo_model.predict(x_test_scaled)
print("=== One-vs-One (OvO) ===")
print(classification_report(y_test, y_pred_ovo))
acc_ovo_rand=accuracy_score(y_test, y_pred_ovo)



=== One-vs-One (OvO) ===
              precision    recall  f1-score   support

      Budget       0.94      0.93      0.93        54
      Luxury       0.97      0.99      0.98        70
     Premium       0.93      0.92      0.92        59
    Standard       0.90      0.91      0.91        69

    accuracy                           0.94       252
   macro avg       0.94      0.93      0.94       252
weighted avg       0.94      0.94      0.94       252



# SMOTE

In [25]:
smote = SMOTE(random_state=42,k_neighbors=1)
X_smote, y_smote = smote.fit_resample(x_train_scaled, y_train)

/Users/asadbek/Documents/hotel_price_category/myenv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [26]:
print("\nSMOTE class distribution:", Counter(y_smote))


SMOTE class distribution: Counter({'Standard': 268, 'Luxury': 268, 'Premium': 268, 'Budget': 268})


# OVR

In [27]:

ovr_model_smote = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
ovr_model_smote.fit(X_smote, y_smote)

# Predict and evaluate
y_pred_ovr_smote = ovr_model_smote.predict(x_test_scaled)
print("=== One-vs-Rest (OvR) ===")
print(classification_report(y_test, y_pred_ovr_smote))
acc_ovr_smote=accuracy_score(y_test, y_pred_ovr_smote)



=== One-vs-Rest (OvR) ===
              precision    recall  f1-score   support

      Budget       0.86      0.89      0.87        54
      Luxury       0.88      1.00      0.93        70
     Premium       0.75      0.61      0.67        59
    Standard       0.74      0.72      0.73        69

    accuracy                           0.81       252
   macro avg       0.80      0.81      0.80       252
weighted avg       0.80      0.81      0.80       252



# OVO

In [28]:

ovo_model_smote = OneVsOneClassifier(LogisticRegression(max_iter=1000, random_state=42))
ovo_model_smote.fit(X_smote, y_smote)


y_pred_ovo_smote = ovo_model_smote.predict(x_test_scaled)
print("=== One-vs-One (OvO) ===")
print(classification_report(y_test, y_pred_ovo_smote))
acc_ovo_smote=accuracy_score(y_test, y_pred_ovo_smote)




=== One-vs-One (OvO) ===
              precision    recall  f1-score   support

      Budget       0.94      0.93      0.93        54
      Luxury       0.97      0.99      0.98        70
     Premium       0.93      0.92      0.92        59
    Standard       0.90      0.91      0.91        69

    accuracy                           0.94       252
   macro avg       0.94      0.93      0.94       252
weighted avg       0.94      0.94      0.94       252



In [29]:
x_train_scaled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1006 entries, 1200 to 1126
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               1006 non-null   float64
 1   price               1006 non-null   float64
 2   adults              1006 non-null   float64
 3   city                1006 non-null   float64
 4   distance_to_center  1006 non-null   float64
 5   description         1006 non-null   float64
 6   bf_has_included     1006 non-null   float64
 7   distance_category   1006 non-null   float64
 8   city_popularity     1006 non-null   float64
 9   adults_category     1006 non-null   float64
dtypes: float64(10)
memory usage: 86.5 KB


In [30]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 1006 entries, 1200 to 1126
Series name: price_category
Non-Null Count  Dtype 
--------------  ----- 
1006 non-null   object
dtypes: object(1)
memory usage: 15.7+ KB


# ADASYN

In [31]:
print("Original class distribution:", Counter(y_train))

Original class distribution: Counter({'Standard': 268, 'Luxury': 257, 'Premium': 255, 'Budget': 226})


In [32]:
adasyn = ADASYN(random_state=42, n_neighbors=1)
X_ada, y_ada = adasyn.fit_resample(x_train_scaled, y_train)

/Users/asadbek/Documents/hotel_price_category/myenv/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


ValueError: No samples will be generated with the provided ratio settings.

In [ ]:
import tabulate

from tabulate import tabulate

table = [
    ["One-vs-One + RandomOverSampler", acc_ovo_rand],
    ["One-vs-One + SMOTE", acc_ovo_smote],
    ["One-vs-Rest + RandomOverSampler", acc_ovr_rand],
    ["One-vs-Rest + SMOTE", acc_ovr_smote],
]

print(tabulate(
    table,
    headers=["Method", "Accuracy"],
    tablefmt="grid",
    floatfmt=".4f"
))

+---------------------------------+------------+
| Method                          |   Accuracy |
+=================================+============+
| One-vs-One + RandomOverSampler  |     0.9365 |
+---------------------------------+------------+
| One-vs-One + SMOTE              |     0.9365 |
+---------------------------------+------------+
| One-vs-Rest + RandomOverSampler |     0.8016 |
+---------------------------------+------------+
| One-vs-Rest + SMOTE             |     0.8095 |
+---------------------------------+------------+
